In [ ]:
import ctypes
try:
    ctypes.CDLL('/usr/lib/aarch64-linux-gnu/libgomp.so.1', mode=ctypes.RTLD_GLOBAL)
except OSError:
    pass

import os
import sys
sys.path.append('/usr/local/lib')
sys.path.append('/usr/local/lib/python3.6/pyrealsense2')

import cv2
import torch
import numpy as np
import math
import time
import csv
import pyrealsense2 as rs
import ipywidgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar
from torch2trt import TRTModule
from utils import preprocess

def quaternion_yaw(qx, qy, qz, qw):
    siny_cosp = 2 * (qw * qy + qz * qx)
    cosy_cosp = 1 - 2 * (qx * qx + qy * qy)
    return math.atan2(siny_cosp, cosy_cosp)

def get_extrinsics(src, dst):
    extrinsics = src.get_extrinsics_to(dst)
    R = np.reshape(list(extrinsics.rotation), (3, 3))
    T = list(extrinsics.translation)
    return R, T

model_trt = TRTModule()
model_trt.load_state_dict(torch.load('road_following_model_merged_0337t_tanh_norbert_laptop_trt.pth'))

car = NvidiaRacecar()
camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True

pipe = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.pose)
config.enable_stream(rs.stream.fisheye, 1)
config.enable_stream(rs.stream.fisheye, 2)
profile = pipe.start(config)

s1 = profile.get_stream(rs.stream.fisheye, 1).as_video_stream_profile()
s2 = profile.get_stream(rs.stream.fisheye, 2).as_video_stream_profile()

intr1 = s1.get_intrinsics()
intr2 = s2.get_intrinsics()
R, T = get_extrinsics(s1, s2)

stereo_fov_rad = 100 * (math.pi / 180)
stereo_size = (320, 240) #<-------------------------- 320x240
stereo_cx = (stereo_size[0] - 1) / 2.0
stereo_cy = (stereo_size[1] - 1) / 2.0
stereo_focal_px = stereo_size[0] / (2 * math.tan(stereo_fov_rad / 2))

P1 = np.array([[stereo_focal_px, 0, stereo_cx, 0], [0, stereo_focal_px, stereo_cy, 0], [0, 0, 1, 0]])
P2 = np.copy(P1)
P2[0, 3] = T[0] * stereo_focal_px

R1 = np.eye(3)
R2 = R

K1 = np.array([[intr1.fx, 0, intr1.ppx], [0, intr1.fy, intr1.ppy], [0, 0, 1]])
D1 = np.array(intr1.coeffs[:4])
K2 = np.array([[intr2.fx, 0, intr2.ppx], [0, intr2.fy, intr2.ppy], [0, 0, 1]])
D2 = np.array(intr2.coeffs[:4])

map1_x, map1_y = cv2.fisheye.initUndistortRectifyMap(K2, D2, R1, P1[:3, :3], stereo_size, cv2.CV_32FC1)
map2_x, map2_y = cv2.fisheye.initUndistortRectifyMap(K1, D1, R2, P2[:3, :3], stereo_size, cv2.CV_32FC1)

window_size = 5 
stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=64,
    blockSize=window_size,
    P1=8 * 1 * window_size**2,
    P2=32 * 1 * window_size**2,
    disp12MaxDiff=1,
    uniquenessRatio=25,
    speckleWindowSize=150, 
    speckleRange=1,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)

print("Sprzęt, kalibracja stereo oraz model TRT zainicjalizowane pomyślnie!")

In [ ]:
camera_widget = widgets.Image(format='jpeg', width=224, height=224)
disparity_widget = widgets.Image(format='jpeg', width=224, height=224)

display(widgets.HBox([camera_widget, disparity_widget]))

In [ ]:
STEERING_GAIN = -1.1
STEERING_BIAS = -0.13
THROTTLE = -0.41 

# omijanie
FRAME_CENTER_X = 112
K_LANE = 1.9                     
MIN_AVOID_TURN = 0.3
AVOID_STEERING_GAIN = 0.01
AVOID_STEERING_BIAS = -0.13
MAX_STEER = 1
MIN_DISPARITY_THRESHOLD = 22.0 #<---zakrycie tła, wiecej
DISP_EVERY_N_FRAMES = 1
WIDGET_EVERY_N_FRAMES = 1

car.throttle = THROTTLE

SAVE_INTERVAL = 0.01
last_save_time = 0
prev_disparity = None
alpha = 0.5
frame_counter = 0
last_rfm_steer = 0.0            

if not os.path.exists('T265_tracking_data.csv'):
    with open('T265_tracking_data.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['X (m)', 'Z (m)', 'YAW (deg)'])

try:
    while True:
        current_time = time.time()
        frames = pipe.poll_for_frames()
        frame_counter += 1

        obstacle_detected = False
        avoid_steering_value = AVOID_STEERING_BIAS

        dynamic_center_x = FRAME_CENTER_X + (last_rfm_steer * FRAME_CENTER_X * K_LANE)
        dynamic_center_x = int(max(0, min(224, dynamic_center_x)))    # <-- NOWE

        if frames and (frame_counter % DISP_EVERY_N_FRAMES == 0):
            f1 = frames.get_fisheye_frame(1)
            f2 = frames.get_fisheye_frame(2)

            if f1 and f2:
                img1 = np.asanyarray(f1.get_data())
                img2 = np.asanyarray(f2.get_data())

                rect1 = cv2.remap(img2, map1_x, map1_y, cv2.INTER_LINEAR)
                rect2 = cv2.remap(img1, map2_x, map2_y, cv2.INTER_LINEAR)

                disp_raw = stereo.compute(rect2, rect1).astype(np.float32) / 16.0
                disp_raw[disp_raw < 0] = 0

                disparity = cv2.medianBlur(disp_raw.astype(np.uint8), 5).astype(np.float32)

                if prev_disparity is None:
                    prev_disparity = disparity
                else:
                    disparity = cv2.addWeighted(disparity, alpha, prev_disparity, 1 - alpha, 0)
                    prev_disparity = disparity

                if frame_counter % WIDGET_EVERY_N_FRAMES == 0:
                    disp_resized_raw = cv2.resize(disparity, (224, 224))

                    disp_near = disp_resized_raw.copy()
                    disp_near[disp_near < MIN_DISPARITY_THRESHOLD] = 0

                    disp_vis = cv2.normalize(disp_near, None, alpha=0, beta=255,
                                              norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
                    disp_resized = cv2.resize(disp_vis, (224, 224))
                    disp_bgr = cv2.cvtColor(disp_resized, cv2.COLOR_GRAY2BGR)

                    valid_disp = disp_near[disp_near > 5]

                    if len(valid_disp) > 0:
                        thresh_val = np.percentile(valid_disp, 85) 
                        _, mask = cv2.threshold(disp_near, thresh_val, 255, cv2.THRESH_BINARY)
                        mask = mask.astype(np.uint8)

                        kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
                        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

                        kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
                        mask = cv2.dilate(mask, kernel_dilate, iterations=2)

                        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                        best_bbox = None
                        max_score = -1

                        for cnt in contours:
                            area = cv2.contourArea(cnt)
                            if area > 600 and area < 4000:
                                x, y, w, h = cv2.boundingRect(cnt)

                                cnt_mask = np.zeros_like(mask)
                                cv2.drawContours(cnt_mask, [cnt], -1, 255, -1)
                                mean_val = cv2.mean(disp_near, mask=cnt_mask)[0]

                                score = mean_val * math.sqrt(area)

                                if score > max_score:
                                    max_score = score
                                    best_bbox = (x, y, w, h)


                        if best_bbox is not None:
                            obstacle_detected = True
                            x, y, w, h = best_bbox
                            obstacle_center_x = x + (w // 2)
                            error_x = obstacle_center_x - dynamic_center_x     

                            turn_direction = 1 if error_x >= 0 else -1

                            proportional_part = error_x * AVOID_STEERING_GAIN
                            kick_part = turn_direction * MIN_AVOID_TURN

                            avoid_steering_value = AVOID_STEERING_BIAS + proportional_part + kick_part
                            avoid_steering_value = max(-MAX_STEER, min(MAX_STEER, avoid_steering_value))

                            cv2.rectangle(disp_bgr, (x, y), (x + w, y + h), (0, 255, 0), 2)
                            cv2.line(disp_bgr, (dynamic_center_x, 0), (dynamic_center_x, 224), (255, 0, 0), 1)   
                    else:
                        cv2.line(disp_bgr, (dynamic_center_x, 0), (dynamic_center_x, 224), (255, 0, 0), 1)       

                    status_text = "AVOID" if obstacle_detected else "FOLLOW"
                    cv2.putText(disp_bgr, status_text, (10, 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

                    disparity_widget.value = bgr8_to_jpeg(disp_bgr)

            pose_frame = frames.get_pose_frame()
            if pose_frame and (current_time - last_save_time >= SAVE_INTERVAL):
                data = pose_frame.get_pose_data()
                x_m = round(data.translation.x, 2)
                z_m = round(-data.translation.z, 2)

                qx, qy, qz, qw = data.rotation.x, data.rotation.y, data.rotation.z, data.rotation.w
                yaw_deg = round(math.degrees(quaternion_yaw(qx, qy, qz, qw)), 2)

                with open('T265_tracking_data.csv', 'a', newline='', encoding='utf-8') as file:
                    writer = csv.writer(file)
                    writer.writerow([x_m, z_m, yaw_deg])

                last_save_time = current_time

        image = camera.value
        if image is None:
            continue

        camera_widget.value = bgr8_to_jpeg(image)

        if obstacle_detected:
            car.steering = avoid_steering_value
        else:
            image_preprocessed = preprocess(image).half()
            output = model_trt(image_preprocessed).detach().cpu().numpy().flatten()
            x = float(output[0])
            car.steering = x * STEERING_GAIN + STEERING_BIAS
            last_rfm_steer = x        

except KeyboardInterrupt:
    print("\nZatrzymano przez użytkownika.")
finally:
    pipe.stop()
    car.throttle = 0.0
    car.steering = STEERING_BIAS
    print("Pojazd zatrzymany, potok T265 zamknięty.")